# Working with parquet files

## Objective

+ In this assignment, we will use the data downloaded with the module `data_manager` to create features.

(11 pts total)

## Prerequisites

+ This notebook assumes that price data is available to you in the environment variable `PRICE_DATA`. If you have not done so, then execute the notebook `01_materials/labs/2_data_engineering.ipynb` to create this data set.


+ Load the environment variables using dotenv. (1 pt)

In [1]:
# Write your code below.
%load_ext dotenv
%dotenv


In [23]:
import dask.dataframe as dd
from glob import glob
import os
import pandas as pd

+ Load the environment variable `PRICE_DATA`.
+ Use [glob](https://docs.python.org/3/library/glob.html) to find the path of all parquet files in the directory `PRICE_DATA`.

(1pt)

In [28]:
# Load the environment variable, PRICE_DATA
# the information for this environment variable is stored in the .env file
price_file = os.getenv("PRICE_DATA")

# Use glob to find the path of all parquet files in the directory PRICE_DATA
price_glob = glob(price_file + '/**/*.parquet', recursive=True)
price_df = dd.read_parquet(price_glob)

# Remove the "Price" as the name of the first column
price_df.columns.name = None
price_df

,Date,Adj Close,Close,High,Low,Open,Volume,Year
npartitions=13078,,,,,,,,
,"datetime64[ns, UTC]",float64,float64,float64,float64,float64,float64,int32
,...,...,...,...,...,...,...,...
...,...,...,...,...,...,...,...,...
,...,...,...,...,...,...,...,...
,...,...,...,...,...,...,...,...


For each ticker and using Dask, do the following:

+ Add lags for variables Close and Adj_Close.
+ Add returns based on Close:
    
    - `returns`: (Close / Close_lag_1) - 1

+ Add the following range: 

    - `hi_lo_range`: this is the day's High minus Low.

+ Assign the result to `dd_feat`.

(4 pt)

In [30]:
# Define a function to apply shift within each partition
def add_lags(df):
    df["Close_lag_1"] = df["Close"].shift(1)
    df["Adj_Close_lag_1"] = df["Adj Close"].shift(1)
    return df

# Apply shift using map_partitions
price_df = price_df.map_partitions(add_lags)

# Create returns column
price_df = price_df.assign(
    returns=(price_df["Close"] / price_df["Close_lag_1"]) - 1,
    hi_lo_range=price_df["High"] - price_df["Low"]
)

# Store the final transformed DataFrame
dd_feat = price_df
dd_feat

,Date,Adj Close,Close,High,Low,Open,Volume,Year,Close_lag_1,Adj_Close_lag_1,returns,hi_lo_range
npartitions=13078,,,,,,,,,,,,
,"datetime64[ns, UTC]",float64,float64,float64,float64,float64,float64,int32,float64,float64,float64,float64
,...,...,...,...,...,...,...,...,...,...,...,...
...,...,...,...,...,...,...,...,...,...,...,...,...
,...,...,...,...,...,...,...,...,...,...,...,...
,...,...,...,...,...,...,...,...,...,...,...,...


+ Convert the Dask data frame to a pandas data frame. 
+ Add a new feature containing the moving average of `returns` using a window of 10 days. There are several ways to solve this task, a simple one uses `.rolling(10).mean()`.

(3 pt)

In [ ]:
# Convert the Dask dataframe to a pandas data frame
dd_feat_pds = dd_feat.compute()

# Add a new feature that contains the average of returns using a window of 10 days
dd_feat_pds["Rolling Average"] = dd_feat_pds["returns"].rolling(10).mean()

In [ ]:
dd_feat_pds

Price,Date,Adj Close,Close,High,Low,Open,Volume,Year,Close_lag_1,Adj_Close_lag_1,returns,hi_lo_range,Rolling Average
Ticker,,,,,,,,,,,,,
A,2000-01-03 00:00:00+00:00,43.382843,51.502148,56.464592,48.193848,56.330471,4674353.0,2000,NaN,NaN,NaN,8.270744,NaN
A,2000-01-04 00:00:00+00:00,40.068878,47.567955,49.266811,46.316166,48.730328,4765083.0,2000,51.502148,43.382843,-0.076389,2.950645,NaN
A,2000-01-05 00:00:00+00:00,37.583393,44.617310,47.567955,43.141991,47.389126,5758642.0,2000,47.567955,40.068878,-0.062030,4.425964,NaN
A,2000-01-06 00:00:00+00:00,36.152363,42.918453,44.349072,41.577251,44.080830,2534434.0,2000,44.617310,37.583393,-0.038076,2.771820,NaN
A,2000-01-07 00:00:00+00:00,39.165077,46.494991,47.165592,42.203148,42.247852,2819626.0,2000,42.918453,36.152363,0.083333,4.962444,NaN
...,...,...,...,...,...,...,...,...,...,...,...,...,...
ZTS,2025-01-27 00:00:00+00:00,173.029999,173.029999,173.479996,168.320007,168.320007,2404500.0,2025,168.610001,168.610001,0.026214,5.159988,0.005888
ZTS,2025-01-28 00:00:00+00:00,170.419998,170.419998,174.929993,169.460007,173.250000,2164600.0,2025,173.029999,173.029999,-0.015084,5.469986,0.002542
ZTS,2025-01-29 00:00:00+00:00,170.220001,170.220001,171.190002,169.000000,169.699997,2144200.0,2025,170.419998,170.419998,-0.001174,2.190002,0.003573


Please comment:

+ Was it necessary to convert to pandas to calculate the moving average return?
+ Would it have been better to do it in Dask? Why?

(1 pt)

It was not neccessary to convert to Pands to calculate the moving average. This is possible to do in dask using operations such as .rolling() without having to do the conversion. It would have been better to do this in Dask because it is more efficient in processing larger data without running into memory limitations.

## Criteria

The [rubric](./assignment_1_rubric_clean.xlsx) contains the criteria for grading.

## Submission Information

🚨 **Please review our [Assignment Submission Guide](https://github.com/UofT-DSI/onboarding/blob/main/onboarding_documents/submissions.md)** 🚨 for detailed instructions on how to format, branch, and submit your work. Following these guidelines is crucial for your submissions to be evaluated correctly.

### Submission Parameters:
* Submission Due Date: `HH:MM AM/PM - DD/MM/YYYY`
* The branch name for your repo should be: `assignment-1`
* What to submit for this assignment:
    * This Jupyter Notebook (assignment_1.ipynb) should be populated and should be the only change in your pull request.
* What the pull request link should look like for this assignment: `https://github.com/<your_github_username>/production/pull/<pr_id>`
    * Open a private window in your browser. Copy and paste the link to your pull request into the address bar. Make sure you can see your pull request properly. This helps the technical facilitator and learning support staff review your submission easily.

Checklist:
- [ ] Created a branch with the correct naming convention.
- [ ] Ensured that the repository is public.
- [ ] Reviewed the PR description guidelines and adhered to them.
- [ ] Verify that the link is accessible in a private browser window.

If you encounter any difficulties or have questions, please don't hesitate to reach out to our team via our Slack at `#cohort-3-help`. Our Technical Facilitators and Learning Support staff are here to help you navigate any challenges.